In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** Funkcje Qiskit to eksperymentalna funkcjonalność dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (za pośrednictwem IBM Quantum Platform API). Mają status wersji podglądowej i mogą ulec zmianie.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>
## Omówienie
W chemii kwantowej problem struktury elektronowej polega na znalezieniu rozwiązań elektronowego równania Schrödingera — kwantowych funkcji falowych opisujących zachowanie elektronów w danym układzie. Funkcje te są wektorami amplitud zespolonych, przy czym każda amplituda odpowiada wkładowi jednej z możliwych konfiguracji elektronowych.

Stan podstawowy to funkcja falowa układu o najniższej energii i odgrywa szczególną rolę w badaniu układów molekularnych. Najbardziej dokładne podejście do obliczania stanu podstawowego uwzględnia wszystkie możliwe konfiguracje elektronowe, lecz dla większych układów staje się niewykonalne, ponieważ liczba konfiguracji rośnie wykładniczo wraz z rozmiarem układu.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) to nowatorska hybrydowa metoda kwantowo-klasyczna służąca do precyzyjnego szacowania stanu podstawowego układów molekularnych. Łączy sprzęt kwantowy z obliczeniami klasycznymi — procesory kwantowe służą do efektywnego przeszukiwania kandydatów na konfiguracje elektronowe, a wynikowa funkcja falowa jest obliczana na komputerach klasycznych. Generując zwarte, lecz chemicznie dokładne funkcje falowe, HI-VQE wspomaga badania i odkrycia w chemii kwantowej oraz nauce o materiałach.

![Obraz przedstawiający ogólny zarys algorytmu HI-VQE firmy Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE zmniejsza złożoność obliczeniową problemu struktury elektronowej, efektywnie szacując stan podstawowy z dużą dokładnością. Skupia się na starannie dobranym podzbiorze najważniejszych konfiguracji elektronowych, optymalizując jednocześnie dokładność i wydajność.

Łącząc zalety komputerów klasycznych i kwantowych, HI-VQE iteracyjnie udoskonala bieżące przybliżenie funkcji falowej. Unikalne techniki budowy podprzestrzeni pomagają zwiększyć efektywność doboru konfiguracji, zapewniając użytkownikom większą kontrolę obliczeniową i lepszą dokładność symulacji chemii kwantowej.

Jeśli chcesz dokładniej poznać algorytm, możesz [przeczytać powiązaną publikację naukową](https://arxiv.org/abs/2503.06292).

## Opis
Liczba konfiguracji elektronowych w układzie molekularnym rośnie wykładniczo wraz z rozmiarem układu. Jednak dla pewnych stanów elektronowych, takich jak stan podstawowy, zazwyczaj tylko niewielki ułamek konfiguracji wnosi istotny wkład do energii stanu. Metody wybranej interakcji konfiguracyjnej (SCI) wykorzystują tę rzadkość do obniżenia kosztów obliczeniowych, identyfikując i koncentrując się na najbardziej istotnych konfiguracjach. Podzbiór tych konfiguracji określa się mianem podprzestrzeni.

HI-VQE wykorzystuje wrodzoną efektywność komputerów kwantowych w reprezentowaniu układów molekularnych, wspomagając przeszukiwanie podprzestrzeni. Integruje klasyczne i kwantowe procedury, aby rozwiązać problem struktury elektronowej z dużą dokładnością. W odróżnieniu od istniejących kwantowych metod SCI, HI-VQE łączy trenowanie wariacjne, iteracyjne budowanie podprzestrzeni oraz wstępne przesiewanie konfiguracji metodą pre-diagonalizacji, co zwiększa efektywność przez ograniczenie liczby pomiarów kwantowych, iteracji i kosztów klasycznej diagonalizacji. HI-VQE można zatem stosować do większych układów molekularnych wymagających większej liczby Qubitów, a koszt rozwiązania problemu danego rozmiaru z tym samym stopniem dokładności ulega zmniejszeniu.

![Obraz przedstawiający szczegółowy opis każdego kroku algorytmu HI-VQE firmy Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Aby obliczyć stan podstawowy układu, HI-VQE najpierw używa klasycznego pakietu chemicznego PySCF do wygenerowania reprezentacji molekularnej na podstawie danych wejściowych podanych przez użytkownika, takich jak geometria molekularna i inne informacje o cząsteczce. Następnie wchodzi w hybrydową pętlę optymalizacji kwantowo-klasycznej, iteracyjnie udoskonalając podprzestrzeń, by optymalnie reprezentować stan podstawowy przy minimalizacji liczby zawartych konfiguracji. Pętla działa do momentu spełnienia kryteriów zbieżności, takich jak rozmiar podprzestrzeni lub stabilność energii; po ich osiągnięciu wynikiem jest obliczona funkcja falowa stanu podstawowego wraz z energią. Wyniki te można wykorzystać do konstruowania dokładnych powierzchni energii potencjalnej i przeprowadzania dalszej analizy układu.

Pętla optymalizacji skupia się na dostosowywaniu parametrów Circuit kwantowego w celu wygenerowania wysokiej jakości podprzestrzeni. HI-VQE oferuje trzy opcje Circuit kwantowego: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) oraz [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Optymalizacja jest inicjalizowana blisko stanu referencyjnego Hartree-Focka ze względu na jego ogólną przydatność. Następnie Circuit jest wykonywany na urządzeniu kwantowym i z wynikowego stanu kwantowego pobierane są próbki konfiguracji, które są zwracane jako ciągi binarne. Ze względu na szumy urządzenia kwantowego niektóre próbkowane konfiguracje mogą być fizycznie nieprawidłowe — nie zachowują liczby elektronów ani spinu. HI-VQE rozwiązuje ten problem, stosując proces odtwarzania konfiguracji z pakietu [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), dzięki czemu użytkownicy mogą albo poprawiać nieprawidłowe konfiguracje, albo je odrzucać.

Prawidłowe konfiguracje przechodzą opcjonalny krok przesiewania, usuwający te, które przewiduje się jako wnoszące minimalny wkład. Zmniejsza to wymiar podprzestrzeni, obniżając tym samym koszt kroku diagonalizacji. Jeśli przesiewanie jest włączone, na podstawie prawidłowych konfiguracji konstruowany jest wstępny hamiltonian podprzestrzeni, a diagonalizacja wykonywana jest z bardzo luźnymi kryteriami zakończenia. Choć dokładność wynikowych amplitud dla każdej konfiguracji jest niska, jest ona skuteczna do przewidywania, które konfiguracje pominąć w podprzestrzeni w danej iteracji, i jest szybka do obliczenia.

Wybrane konfiguracje są dodawane do podprzestrzeni, a hamiltonian układu jest rzutowany do tej podprzestrzeni. Podprzestrzeń aktualizuje się iteracyjnie, zachowując najważniejsze konfiguracje pomiędzy iteracjami. Podejście to różni się od metod alternatywnych tym, że Circuit kwantowy nie musi przybliżać pełnego stanu podstawowego w każdym kroku.

Następnie hamiltonian podprzestrzeni jest klasycznie diagonalizowany w celu uzyskania najniższej wartości własnej i odpowiadającego jej wektora własnego, reprezentującego przybliżenie stanu podstawowego i jego energii. W miarę jak jakość podprzestrzeni poprawia się w kolejnych iteracjach, obliczony stan podstawowy coraz lepiej przybliża rzeczywisty stan podstawowy. Na tym etapie można przeprowadzić dodatkowy krok przesiewania, usuwający z podprzestrzeni wszelkie konfiguracje, które nie wnoszą istotnego wkładu do obliczonego stanu podstawowego. Krok ten zapewnia, że podprzestrzeń przenoszona do następnej iteracji jest możliwie jak najbardziej zwarta. Ocena odbywa się na podstawie amplitud zwróconych przez diagonalizację, gdyż reprezentują one ważność wkładu każdej konfiguracji do obliczonego stanu podstawowego.

Sprawdzenie zbieżności rozstrzyga, czy dalsze trenowanie mogłoby poprawić wyniki. Jeśli tak, wykonywany jest opcjonalny krok klasycznej ekspansji, parametry Circuit kwantowego są aktualizowane w celu dalszej minimalizacji obliczonej energii, a proces się powtarza. Krok klasycznej ekspansji generuje dodatkowe konfiguracje dla podprzestrzeni, uzupełniając konfiguracje próbkowane z urządzenia kwantowego. Najpierw identyfikuje konfigurację o największej amplitudzie w wynikach diagonalizacji, a następnie generuje nowe konfiguracje z pojedynczymi i podwójnymi wzbudzeniami z tej konfiguracji. Żądana liczba tych konfiguracji jest następnie dodawana do podprzestrzeni.

Gdy zostanie stwierdzona zbieżność iteracji, HI-VQE zwraca obliczony stan podstawowy (w postaci stanów w podprzestrzeni i ich amplitud w funkcji falowej stanu podstawowego), jego energię oraz miarę wariancji energii, która wskazuje, czy obliczony stan stanowi stan własny hamiltonianu układu.

Użytkownicy mogą decydować o używanym Circuit kwantowym i liczbie strzałów (shots) dla każdego Circuit kwantowego, a także kontrolować rozmiar podprzestrzeni lub włączyć klasyczne generowanie dodatkowych konfiguracji wspomagających konfiguracje generowane kwantowo. Dzięki temu użytkownicy mogą dostosować zachowanie HI-VQE do swoich docelowych zastosowań.

## Licencjonowanie
Pamiętaj, że korzystanie z tej funkcji Qiskit jest ograniczone do problemów wymagających co najwyżej 20 Qubitów, chyba że uzyskano licencję przyznającą wyższy limit.

Jeśli chcesz uzyskać informacje na temat licencji, napisz na adres [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com).
## Pierwsze kroki
Najpierw [poproś o dostęp do funkcji](https://forms.office.com/r/zN3hvMTqJ1).
Następnie uwierzytelnij się przy użyciu swojego [klucza API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) i — zakładając, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w środowisku lokalnym — załaduj funkcję Qiskit w następujący sposób:

In [2]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Przykład

Pierwszy przykład pokazuje, jak obliczyć energię stanu podstawowego cząsteczki NH3 przy użyciu algorytmu HI-VQE.

#### Zdefiniuj geometrię molekularną i opcje

Geometria molekularna NH3 jest podana we współrzędnych kartezjańskich oddzielonych znakiem ";" dla każdego atomu.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Dodatkowe opcje można zdefiniować i przekazać dla układu molekularnego w następującym formacie słownika.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Uruchom funkcję, podając geometrię i opcje jako dane wejściowe.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Dobrym pomysłem jest wydrukowanie identyfikatora zadania Function, aby móc go podać w zgłoszeniu do pomocy technicznej, jeśli coś pójdzie nie tak.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Ten przykład wykorzystuje 16 Qubitów z 8 orbitalami bazy sto3g dla cząsteczki NH3.
Sprawdź [status](/guides/functions#check-job-status) zadania Qiskit Function lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [9]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


Po zakończeniu zadania wyniki można uzyskać za pomocą instancji `result()`.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Aby uzyskać dostęp do energii stanu podstawowego, użyj klucza `"energy"`. Klucz `"eigenvector"` dostarcza współczynniki CI wraz z odpowiadającą im notacją bitstring konfiguracji elektronowej, przechowywaną pod kluczem `"states"` wyników.

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Wydajność

W tej sekcji przedstawiono wyniki wzorcowych obliczeń HI-VQE: przypadek 24-Qubitowy dla Li2S, przypadek 40-Qubitowy dla cząsteczki N2 oraz przypadek 44-Qubitowy dla układu FeP-NO.

#### Krzywa potencjalnej powierzchni energii dysocjacji dla cząsteczki Li2S z 24 Qubitami

Krzywa PES jest pokazana wraz z referencją FCI i początkowym przybliżeniem z RHF, a także błędem energii względem referencji FCI.

![Obraz pokazujący, że HI-VQE daje rozwiązania mieszczące się w dokładności chemicznej klasycznej referencyjnej krzywej PES dla układu Li2S](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

Obliczenia zostały przeprowadzone dla następujących geometrii i opcji.